In [ ]:
%cd ..

In [ ]:
from pathlib import Path
from mol_gen_docking.data.pydantic_dataset import read_jsonl
import json

In [ ]:
molgendata= Path("data/molgendata/test_data/test_prompts_ood.jsonl")
sys_prompt = json.load(open("system_prompts/vanilla.json"))

data = read_jsonl(molgendata)

In [ ]:
data = data[:1]

In [ ]:
def sample_to_request(sample, max_tokens: int=48, temperature: float=0.7, reasoning_effort: str = "high", n=1):
    id = sample.identifier
    messages = [
        sys_prompt,
        {
            "role": "user", "content": sample.conversations[0].messages[-1].content
        }
    ]
    return dict(
        custom_id = id,
        body = dict(
            max_tokens = max_tokens,
            temperature = temperature,
            reasoning_effort = reasoning_effort,
            n=n,
            messages=messages
        )
    )


requests = [sample_to_request(sample) for sample in data]

In [ ]:
requests

In [ ]:
# save as a jsonl file
with open("notebooks/molgendata_ms_batch_api.jsonl", "w") as f:
    for request in requests:
        f.write(json.dumps(request) + "\n")

In [ ]:
from mistralai.client import Mistral
import os

In [ ]:
# api_key = os.environ["MISTRAL_API_KEY"]
client = Mistral(api_key="GS5AJdn7VSjTGOflo7KncO4vkHbuLtk9")

for m in client.models.list().data:
    if m.name and "small" in m.name.lower():
        print(f"ID: {m.id} | Name: {m.name} | aliases: {';'.join(m.aliases)}")

In [ ]:
batch_data = client.files.upload(
    file={
        "file_name": "test.jsonl",
        "content": open("notebooks/molgendata_ms_batch_api.jsonl", "rb")
    },
    purpose = "batch"
)

In [ ]:
created_job = client.batch.jobs.create(
    input_files=[batch_data.id],
    model="mistral-small-latest",
    endpoint="/v1/chat/completions",
    metadata={"job_type": "testing"}
)

In [14]:
retrieved_job = client.batch.jobs.get(job_id=created_job.id)

if retrieved_job.status == "QUEUED":
    print("Waiting for job to complete...")
else:
    try:
        output_file_stream = client.files.download(file_id=retrieved_job.error_file)
        error = json.load(output_file_stream)["response"]["body"]
        print(json.loads(error)["message"])
    except Exception as e:
        print("No errors")

The model `mistral-small-2603` does not exist for router `completion`.


In [ ]:
retrieved_job

In [ ]:
# Write and save the file
with open('notebooks/batch_results.jsonl', 'wb') as f:
    f.write(output_file_stream.read())